# Production Patterns with Medha

This notebook covers the patterns you need when deploying Medha in a real application:

1. **`store_batch()`** — populate the cache from large datasets in a single round-trip
2. **`warm_from_file()`** — initialize the cache from a JSON/JSONL export at startup
3. **Persistent embedding cache** — avoid recomputing embeddings across restarts
4. **Context manager** — `async with Medha(...)` for clean lifecycle management
5. **Redis L1 cache** — share the hot-path cache across multiple service instances
6. **`stats` dashboard** — monitor hit rates and per-tier latency in real time

**Requirements:**
```bash
pip install medha[fastembed]          # core + local embeddings
pip install medha[redis]              # optional: Redis L1 cache
```

## Cell 1: Imports

In [ ]:
import asyncio
import json
import os
import tempfile
import time

from medha import Medha, Settings, InMemoryL1Cache
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import QueryTemplate

## Cell 2: `store_batch()` — Bulk Cache Population

`store_batch()` computes all embeddings in a **single call** to the embedder,
then upserts all entries at once. This is much faster than calling `store()` in
a loop for large datasets.

Each entry is a dict with keys:
- `question` *(required)*
- `generated_query` *(required)*
- `response_summary` *(optional)*
- `template_id` *(optional)*

In [ ]:
entries = [
    {"question": "How many users are there?",        "generated_query": "SELECT COUNT(*) FROM users"},
    {"question": "List all active users",             "generated_query": "SELECT * FROM users WHERE status = 'active'"},
    {"question": "Show the top 10 products by price", "generated_query": "SELECT * FROM products ORDER BY price DESC LIMIT 10"},
    {"question": "What is the average order value?",  "generated_query": "SELECT AVG(total) FROM orders"},
    {"question": "Find all pending orders",           "generated_query": "SELECT * FROM orders WHERE status = 'pending'"},
    {"question": "Count employees by department",     "generated_query": "SELECT department, COUNT(*) FROM employees GROUP BY department"},
    {"question": "Get revenue for last month",        "generated_query": "SELECT SUM(total) FROM orders WHERE created_at >= date('now','-1 month')"},
    {"question": "List products out of stock",        "generated_query": "SELECT * FROM products WHERE stock = 0"},
]

embedder = FastEmbedAdapter()
settings = Settings(qdrant_mode="memory")
medha = Medha("prod_demo", embedder=embedder, settings=settings)
await medha.start()

start = time.perf_counter()
ok = await medha.store_batch(entries)
elapsed = (time.perf_counter() - start) * 1000

print(f"store_batch({len(entries)} entries): {'OK' if ok else 'FAILED'} in {elapsed:.1f}ms")
print(f"Equivalent to {len(entries)} store() calls, but uses 1 embedding round-trip")

await medha.close()

## Cell 3: `warm_from_file()` — Cache Initialization at Startup

In production you typically export question-query pairs to a file (from your
LLM logs, golden dataset, or manual curation) and load them at startup.

`warm_from_file()` supports both **JSON arrays** and **JSONL** (one object per line).

In [ ]:
# Create a sample warm-up file (JSON array format)
warm_data = [
    {"question": "How many users are registered?",    "generated_query": "SELECT COUNT(*) FROM users"},
    {"question": "Show all premium subscribers",       "generated_query": "SELECT * FROM users WHERE plan = 'premium'"},
    {"question": "Top 5 bestselling products",         "generated_query": "SELECT * FROM products ORDER BY sales DESC LIMIT 5"},
    {"question": "Total revenue this year",            "generated_query": "SELECT SUM(total) FROM orders WHERE YEAR(created_at) = YEAR(NOW())"},
    {"question": "Overdue invoices count",             "generated_query": "SELECT COUNT(*) FROM invoices WHERE due_date < NOW() AND paid = 0"},
]

# Write to a temporary file to simulate a real startup scenario
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(warm_data, f)
    warm_file = f.name

print(f"Warm-up file created: {warm_file}")
print(f"Contents preview: {len(warm_data)} entries\n")

# Initialize Medha and warm the cache in one call
embedder = FastEmbedAdapter()
settings = Settings(qdrant_mode="memory")
medha = Medha("warm_demo", embedder=embedder, settings=settings)
await medha.start()

start = time.perf_counter()
stored = await medha.warm_from_file(warm_file)
elapsed = (time.perf_counter() - start) * 1000

print(f"warm_from_file(): {stored} entries loaded in {elapsed:.1f}ms")

# Verify — search for one of the loaded questions
hit = await medha.search("How many users are registered?")
print(f"\nSearch test: strategy={hit.strategy}, query={hit.generated_query!r}")

await medha.close()
os.unlink(warm_file)

## Cell 4: Persistent Embedding Cache

By default, Medha recomputes embeddings on every cold start. With
`embedding_cache_path` set, embeddings are:
- **Saved to disk** on `close()`
- **Loaded from disk** on the next `start()`

This significantly reduces startup time when the same questions recur across sessions.

In [ ]:
cache_file = tempfile.mktemp(suffix="_embeddings.json")

pairs = [
    ("Count total orders",       "SELECT COUNT(*) FROM orders"),
    ("List active customers",    "SELECT * FROM customers WHERE active = 1"),
    ("Average product rating",   "SELECT AVG(rating) FROM reviews"),
]

# --- Session 1: cold start, embeddings computed from scratch ---
print("Session 1 (cold start)...")
embedder = FastEmbedAdapter()
settings = Settings(qdrant_mode="memory", embedding_cache_path=cache_file)
medha = Medha("emb_cache_demo", embedder=embedder, settings=settings)

start = time.perf_counter()
await medha.start()
for q, sql in pairs:
    await medha.store(q, sql)
t_cold = (time.perf_counter() - start) * 1000

await medha.close()  # Saves embeddings to disk
print(f"  Session 1 duration: {t_cold:.1f}ms")
print(f"  Embedding cache saved to: {cache_file}")
print(f"  Cache file size: {os.path.getsize(cache_file):,} bytes\n")

# --- Session 2: warm start, embeddings loaded from disk ---
print("Session 2 (warm start — embeddings from disk)...")
embedder2 = FastEmbedAdapter()
medha2 = Medha("emb_cache_demo", embedder=embedder2, settings=settings)

start = time.perf_counter()
await medha2.start()  # Loads embeddings from disk
for q, sql in pairs:
    await medha2.store(q, sql)
t_warm = (time.perf_counter() - start) * 1000

print(f"  Session 2 duration: {t_warm:.1f}ms")
print(f"  Embedding cache size: {medha2.stats['embedding_cache_size']} entries")

await medha2.close()
os.unlink(cache_file)

## Cell 5: Context Manager — Clean Lifecycle

`Medha` implements `__aenter__` / `__aexit__`, so you can use it as an async
context manager. This guarantees that `start()` and `close()` are always called,
even if an exception occurs — exactly what you want in application startup/shutdown hooks.

In [ ]:
embedder = FastEmbedAdapter()
settings = Settings(qdrant_mode="memory")

async with Medha("ctx_demo", embedder=embedder, settings=settings) as medha:
    await medha.store("Total active sessions", "SELECT COUNT(*) FROM sessions WHERE active = 1")
    await medha.store("Daily new signups",      "SELECT COUNT(*) FROM users WHERE DATE(created_at) = CURDATE()")

    hit = await medha.search("How many sessions are currently active?")
    print(f"Hit: strategy={hit.strategy}")
    print(f"     query={hit.generated_query}")
    print(f"     confidence={hit.confidence:.4f}")

# medha.close() was called automatically on __aexit__
print("\nContext exited cleanly — close() called automatically.")

## Cell 6: Redis L1 Cache — Shared Hot Path

The default `InMemoryL1Cache` lives inside a single process. When you scale
horizontally (multiple pods/workers), each instance has its own cold L1 cache.

`RedisL1Cache` solves this: all instances share a single Redis namespace, so
a cache hit warmed by instance A is immediately available to instance B.

> **Tip:** Configure `maxmemory-policy allkeys-lru` on Redis for automatic eviction.

In [ ]:
REDIS_URL = "redis://localhost:6379/0"  # Change to your Redis URL

try:
    from medha import RedisL1Cache
    import redis.asyncio as aioredis

    # Probe Redis availability
    _client = aioredis.from_url(REDIS_URL)
    await _client.ping()
    await _client.aclose()

    redis_l1 = RedisL1Cache(
        url=REDIS_URL,
        prefix="medha:demo",  # namespace — avoids key collisions with other apps
        ttl=3600,             # entries expire after 1 hour
    )

    embedder = FastEmbedAdapter()
    settings = Settings(qdrant_mode="memory")

    async with Medha("redis_demo", embedder=embedder, settings=settings, l1_backend=redis_l1) as medha:
        await medha.store("Count all products", "SELECT COUNT(*) FROM products")

        # First search — goes through vector store, result stored in Redis
        hit1 = await medha.search("Count all products")
        print(f"First search:  strategy={hit1.strategy}")

        # Second search — served from Redis L1
        hit2 = await medha.search("Count all products")
        print(f"Second search: strategy={hit2.strategy}")

    print("\nRedis L1 cache demo completed.")

except Exception as e:
    print(f"Redis not available ({e}).")
    print("Running with InMemoryL1Cache instead (single-process equivalent).\n")

    embedder = FastEmbedAdapter()
    settings = Settings(qdrant_mode="memory")

    async with Medha("inmem_demo", embedder=embedder, settings=settings) as medha:
        await medha.store("Count all products", "SELECT COUNT(*) FROM products")

        hit1 = await medha.search("Count all products")
        print(f"First search:  strategy={hit1.strategy}")

        hit2 = await medha.search("Count all products")
        print(f"Second search: strategy={hit2.strategy} (L1 hit)")

## Cell 7: Stats Dashboard

`medha.stats` returns a snapshot of all performance counters:

| Key | Description |
|-----|-------------|
| `total_requests` | All search calls since start |
| `hit_rate` | Percentage of non-miss, non-error results |
| `by_strategy` | Per-tier hit counts |
| `tier_latencies_ms` | Average, total, and call count per tier |
| `l1_cache_size` | Current L1 entries in memory |
| `embedding_cache_size` | In-process embedding LRU entries |
| `total_stored` | Entries stored in this session |
| `templates_loaded` | Templates synced to backend |

In [ ]:
import json as _json

embedder = FastEmbedAdapter()
settings = Settings(qdrant_mode="memory")

async with Medha("stats_demo", embedder=embedder, settings=settings) as medha:
    # Populate
    await medha.store_batch([
        {"question": "Total users",            "generated_query": "SELECT COUNT(*) FROM users"},
        {"question": "Active subscriptions",   "generated_query": "SELECT COUNT(*) FROM subscriptions WHERE active=1"},
        {"question": "Revenue last 30 days",   "generated_query": "SELECT SUM(amount) FROM payments WHERE created_at > NOW()-INTERVAL 30 DAY"},
    ])

    # Simulate a mixed workload
    queries = [
        "Total users",                          # exact / L1 on repeat
        "Total users",                          # L1 hit
        "How many users do we have?",           # semantic
        "What is the user count?",              # semantic
        "Active subscriptions",                 # exact
        "Something completely unrelated",       # miss
    ]
    for q in queries:
        await medha.search(q)

    stats = medha.stats

print("=== Medha Stats ===")
print(f"Total requests : {stats['total_requests']}")
print(f"Hit rate       : {stats['hit_rate']:.1f}%")
print(f"L1 cache size  : {stats['l1_cache_size']} entries")
print(f"Emb cache size : {stats['embedding_cache_size']} entries")
print(f"Total stored   : {stats['total_stored']}")
print()
print("Hits by strategy:")
for strategy, count in stats["by_strategy"].items():
    print(f"  {strategy:20s} : {count}")
print()
print("Tier latencies (avg ms):")
for tier, data in stats["tier_latencies_ms"].items():
    if data["calls"] > 0:
        print(f"  {tier:12s} : {data['avg_ms']:7.3f}ms  ({data['calls']} calls)")

## Cell 8: Putting It All Together — Production Startup Pattern

A realistic production application startup:
1. Load settings from environment variables
2. Initialize with persistent embedding cache
3. Warm from a curated dataset file
4. Use context manager for lifecycle management
5. Monitor stats periodically

In [ ]:
# Simulate env-driven configuration (in real apps, use os.environ or .env)
warm_entries = [
    {"question": q, "generated_query": sql} for q, sql in [
        ("Total registered users",       "SELECT COUNT(*) FROM users"),
        ("Monthly recurring revenue",    "SELECT SUM(amount) FROM subscriptions WHERE active=1"),
        ("Open support tickets",         "SELECT COUNT(*) FROM tickets WHERE status='open'"),
        ("Products with low stock",      "SELECT * FROM products WHERE stock < 10"),
        ("New users this week",          "SELECT COUNT(*) FROM users WHERE created_at > NOW()-INTERVAL 7 DAY"),
    ]
]

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(warm_entries, f)
    startup_file = f.name

emb_cache = tempfile.mktemp(suffix="_emb.json")

settings = Settings(
    qdrant_mode="memory",          # Use "docker" or "cloud" in production
    embedding_cache_path=emb_cache,
    l1_cache_max_size=5000,
)

print("Starting Medha with production configuration...")
async with Medha("app_cache", embedder=FastEmbedAdapter(), settings=settings) as medha:
    n = await medha.warm_from_file(startup_file)
    print(f"  Cache warmed: {n} entries loaded")

    # Simulate application queries
    test_queries = [
        "How many users have signed up?",
        "Total registered users",        # exact repeat
        "What is our MRR?",
        "How many open tickets do we have?",
        "Show products that are running low on inventory",
    ]
    print("\nSimulating application queries:")
    for q in test_queries:
        hit = await medha.search(q)
        print(f"  [{hit.strategy.value:18s}] {q!r}")
        if hit.generated_query:
            print(f"    → {hit.generated_query}")

    stats = medha.stats
    print(f"\nFinal hit rate: {stats['hit_rate']:.1f}%")

for f in (startup_file, emb_cache):
    if os.path.exists(f):
        os.unlink(f)

print("\nApplication shutdown complete.")